In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import time
import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
DATA_DIR = "dataset"  # Path to your dataset directory
OUTPUT_DIR = "10_Grad_CAM_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Classes
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls for idx, cls in enumerate(CLASSES)}

# ===================== DATASET AND DATA LOADING ===================== #
class BrainTumorDataset(Dataset):
    def __init__(self, data_dir, split='Training', transform=None):
        self.data_dir = data_dir
        self.split = split
        self.transform = transform
        
        if self.split not in ['Training', 'Testing']:
            raise ValueError("Split must be one of 'Training' or 'Testing'")
        
        # Path to split folder
        self.split_dir = os.path.join(data_dir, split)
        
        # Get list of all image paths and their labels
        self.image_paths = []
        self.labels = []
        
        for class_name in CLASSES:
            class_dir = os.path.join(self.split_dir, class_name)
            if not os.path.exists(class_dir):
                print(f"Warning: {class_dir} does not exist")
                continue
                
            for img_path in glob.glob(os.path.join(class_dir, "*.jpg")) + glob.glob(os.path.join(class_dir, "*.png")):
                self.image_paths.append(img_path)
                self.labels.append(CLASS_TO_IDX[class_name])
        
        print(f"Loaded {len(self.image_paths)} {split} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Read image using OpenCV
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        
        # Apply transformations
        if self.transform:
            img = self.transform(img)
        
        return img, label, img_path

# Define image transformations
def get_transforms(split):
    if split == 'Training':
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:  # Testing
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ===================== MODEL DEFINITION ===================== #
class BrainTumorClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super(BrainTumorClassifier, self).__init__()
        # Load pre-trained ResNet50
        self.model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # Modify the final fully connected layer
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, num_classes)
        
        # Save the layer names for Grad-CAM
        self.gradients = None
        self.activations = None
        
        # Register hook for the last convolutional layer
        self.model.layer4[-1].conv3.register_forward_hook(self.save_activation)
        self.model.layer4[-1].conv3.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
    
    def forward(self, x):
        return self.model(x)
    
    def get_activations_gradient(self):
        return self.gradients
    
    def get_activations(self):
        return self.activations

# ===================== TRAINING FUNCTIONS ===================== #
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25):
    """Train the model and evaluate on validation set after each epoch"""
    best_acc = 0.0
    best_model_wts = model.state_dict()
    
    # Training metrics
    train_loss_history = []
    train_acc_history = []
    val_loss_history = []
    val_acc_history = []
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0
        
        # Iterate over data
        for inputs, labels, _ in tqdm(train_loader, desc="Training"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        
        if scheduler is not None:
            scheduler.step()
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)
        
        train_loss_history.append(epoch_loss)
        train_acc_history.append(epoch_acc.cpu().item())
        
        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        # Validation phase
        model.eval()
        running_loss = 0.0
        running_corrects = 0
        
        with torch.no_grad():
            for inputs, labels, _ in tqdm(val_loader, desc="Validation"):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                
                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
        epoch_loss = running_loss / len(val_loader.dataset)
        epoch_acc = running_corrects.double() / len(val_loader.dataset)
        
        val_loss_history.append(epoch_loss)
        val_acc_history.append(epoch_acc.cpu().item())
        
        print(f'Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        # Save the best model
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = model.state_dict()
    
    print(f'Best Val Acc: {best_acc:.4f}')
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    
    return model, {
        'train_loss': train_loss_history,
        'train_acc': train_acc_history,
        'val_loss': val_loss_history,
        'val_acc': val_acc_history
    }

def evaluate_model(model, test_loader):
    """Evaluate the model on the test set"""
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels, _ in tqdm(test_loader, desc="Testing"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc:.4f}")
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=CLASSES))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(CLASSES))
    plt.xticks(tick_marks, CLASSES, rotation=45)
    plt.yticks(tick_marks, CLASSES)
    
    # Add text annotations to the confusion matrix
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    horizontalalignment="center",
                    color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.pdf'), format='pdf', bbox_inches='tight')
    plt.close()
    
    return acc, cm

# ===================== GRAD-CAM IMPLEMENTATION ===================== #
def gradcam(model, input_image, target_class=None):
    """Generate Grad-CAM visualization for the input image"""
    # Set model to evaluation mode
    model.eval()
    
    # Forward pass
    output = model(input_image)
    
    # If target class is None, use the predicted class
    if target_class is None:
        target_class = output.argmax(dim=1).item()
    
    # Zero all existing gradients
    model.zero_grad()
    
    # Backward pass with the target class
    output[0, target_class].backward()
    
    # Get gradients and activations
    gradients = model.get_activations_gradient()
    activations = model.get_activations()
    
    # Global average pooling
    weights = torch.mean(gradients, dim=[2, 3])[0, :]
    
    # Create class activation map
    cam = torch.zeros(activations.shape[2:], dtype=torch.float32).to(device)
    
    # Weight the channels by corresponding gradients
    for i, w in enumerate(weights):
        cam += w * activations[0, i, :, :]
    
    # Apply ReLU
    cam = F.relu(cam)
    
    # Normalize
    cam = cam - torch.min(cam)
    if torch.max(cam) > 0:
        cam = cam / torch.max(cam)
    
    return cam.cpu().detach().numpy()

def show_gradcam(image_path, model, target_class=None, transform=None):
    """Show Grad-CAM visualization for an image"""
    # Load image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Prepare image for model
    if transform:
        input_tensor = transform(img).unsqueeze(0).to(device)
    else:
        # If no transform provided, use a basic one
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        input_tensor = transform(img).unsqueeze(0).to(device)
    
    # Generate Grad-CAM for specified class or all classes
    classes_to_visualize = [target_class] if target_class is not None else range(len(CLASSES))
    results = []
    
    for class_idx in classes_to_visualize:
        # Get Grad-CAM for current class
        cam = gradcam(model, input_tensor, class_idx)
        
        # Resize CAM to match image size
        cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
        
        # Convert CAM to heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        
        # Combine original image with heatmap
        img_float = img.astype(float) / 255
        heatmap_float = heatmap.astype(float) / 255
        result = 0.7 * img_float + 0.3 * heatmap_float
        result = np.clip(result, 0, 1)
        results.append((cam_resized, result, class_idx))
    
    return results, img

# Function to generate Grad-CAM visualizations for all classes
def generate_gradcam_visualizations(model, test_loader, num_samples=5):
    """Generate Grad-CAM visualizations for a set of test images for all classes"""
    model.eval()
    test_transform = get_transforms('Testing')
    
    # Get samples from each class
    class_samples = {cls: [] for cls in CLASSES}
    
    with torch.no_grad():
        for inputs, labels, img_paths in test_loader:
            for img, label, path in zip(inputs, labels, img_paths):
                class_name = IDX_TO_CLASS[label.item()]
                if len(class_samples[class_name]) < num_samples:
                    class_samples[class_name].append(path)
                
                # Break if we have enough samples for each class
                if all(len(samples) >= num_samples for samples in class_samples.values()):
                    break
    
    # Create directory for saving visualizations
    os.makedirs(os.path.join(OUTPUT_DIR, 'gradcam'), exist_ok=True)
    
    # Generate and save Grad-CAM visualizations for each sample and class
    for class_name, img_paths in class_samples.items():
        print(f"Generating Grad-CAM visualizations for class: {class_name}")
        for i, img_path in enumerate(img_paths):
            # Generate Grad-CAM for all classes
            results, original_img = show_gradcam(img_path, model, target_class=None, transform=test_transform)
            
            # Create figure to display all Grad-CAM results
            plt.figure(figsize=(15, 10))
            plt.subplot(2, 3, 1)
            plt.imshow(original_img)
            plt.title(f"Original Image\nTrue Class: {class_name}")
            plt.axis('off')
            
            for j, (_, result, class_idx) in enumerate(results):
                plt.subplot(2, 3, j+2)
                plt.imshow(result)
                plt.title(f"Grad-CAM for: {IDX_TO_CLASS[class_idx]}")
                plt.axis('off')
            
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam', f"{class_name}_sample_{i+1}_gradcam.png"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam', f"{class_name}_sample_{i+1}_gradcam.pdf"), format='pdf', bbox_inches='tight')
            plt.close()
    
    print("Grad-CAM visualizations generated successfully!")

# ===================== MAIN EXECUTION ===================== #
def main():
    # Create data transforms
    train_transform = get_transforms('Training')
    test_transform = get_transforms('Testing')
    
    # Create datasets
    train_dataset = BrainTumorDataset(DATA_DIR, split='Training', transform=train_transform)
    test_dataset = BrainTumorDataset(DATA_DIR, split='Testing', transform=test_transform)
    
    # Since we don't have a separate validation set, we'll create one from the training data
    # Use 80% for training and 20% for validation
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
    )
    
    # Create dataloaders
    batch_size = 32
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Define model, loss function, optimizer, and scheduler
    model = BrainTumorClassifier(num_classes=len(CLASSES)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
    
    # Check if saved model exists
    model_path = os.path.join(OUTPUT_DIR, 'brain_tumor_model.pth')
    if os.path.exists(model_path):
        print(f"Loading pre-trained model from {model_path}")
        model.load_state_dict(torch.load(model_path, map_location=device))
    else:
        print("Training model...")
        model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=20
        )
        
        # Save model
        torch.save(model.state_dict(), model_path)
        
        # Plot training curves
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        plt.subplot(1, 2, 2)
        plt.plot(history['train_acc'], label='Train Accuracy')
        plt.plot(history['val_acc'], label='Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.pdf'), format='pdf', bbox_inches='tight')
        plt.close()
    
    # Evaluate model on test set
    print("Evaluating model on test set...")
    test_acc, confusion_mat = evaluate_model(model, test_loader)
    
    # Generate Grad-CAM visualizations
    print("Generating Grad-CAM visualizations...")
    generate_gradcam_visualizations(model, test_loader, num_samples=5)
    
    print("Done!")

if __name__ == "__main__":
    main()

Using device: cpu
Loaded 5712 Training images
Loaded 1311 Testing images
Training model...
Epoch 1/20
----------


Training: 100%|██████████| 143/143 [14:49<00:00,  6.22s/it]


Train Loss: 0.4522 Acc: 0.8345


Validation: 100%|██████████| 36/36 [02:11<00:00,  3.64s/it]


Val Loss: 0.5104 Acc: 0.8215
Epoch 2/20
----------


...

Train Loss: 0.0189 Acc: 0.9947


Validation: 100%|██████████| 36/36 [02:10<00:00,  3.62s/it]


Val Loss: 0.0602 Acc: 0.9799
Best Val Acc: 0.9851
Evaluating model on test set...


Testing: 100%|██████████| 41/41 [02:32<00:00,  3.71s/it]


Test Accuracy: 0.9916

Classification Report:
              precision    recall  f1-score   support

      glioma       1.00      0.98      0.99       300
  meningioma       0.97      0.99      0.98       306
     notumor       1.00      1.00      1.00       405
   pituitary       1.00      1.00      1.00       300

    accuracy                           0.99      1311
   macro avg       0.99      0.99      0.99      1311
weighted avg       0.99      0.99      0.99      1311

Generating Grad-CAM visualizations...
Generating Grad-CAM visualizations for class: glioma
Generating Grad-CAM visualizations for class: meningioma
Generating Grad-CAM visualizations for class: notumor
Generating Grad-CAM visualizations for class: pituitary
Grad-CAM visualizations generated successfully!
Done!


In [5]:
def generate_gradcam_for_image(image_path, model_path=None, output_dir="gradcam_results", save_results=True):
    import os
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    import torch.nn.functional as F
    import torchvision.transforms as transforms
    from torchvision.models import resnet50
    
    # Create output directory if it doesn't exist and save_results is True
    if save_results:
        os.makedirs(output_dir, exist_ok=True)
    
    # Define classes
    CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
    
    # Set device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Define transformation for the input image
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Create a copy of the original image for display
    original_img = img.copy()
    
    # Apply transformations to prepare for model
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    # Define the model with Grad-CAM hooks
    class BrainTumorClassifier(torch.nn.Module):
        def __init__(self, num_classes=4):
            super(BrainTumorClassifier, self).__init__()
            # Load pre-trained ResNet50
            self.model = resnet50(weights=None)  # No pre-trained weights here
            
            # Modify the final fully connected layer
            num_features = self.model.fc.in_features
            self.model.fc = torch.nn.Linear(num_features, num_classes)
            
            # Save the layer names for Grad-CAM
            self.gradients = None
            self.activations = None
            
            # Register hook for the last convolutional layer
            self.model.layer4[-1].conv3.register_forward_hook(self.save_activation)
            self.model.layer4[-1].conv3.register_full_backward_hook(self.save_gradient)
        
        def save_activation(self, module, input, output):
            self.activations = output
        
        def save_gradient(self, module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        def forward(self, x):
            return self.model(x)
        
        def get_activations_gradient(self):
            return self.gradients
        
        def get_activations(self):
            return self.activations
    
    # Load the model
    model = BrainTumorClassifier(num_classes=len(CLASSES)).to(device)
    
    if model_path is None:
        model_path = os.path.join("10_Grad_CAM_Results", "brain_tumor_model.pth")
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Get model prediction
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)[0]
        predicted_class = output.argmax(dim=1).item()
    
    # Print prediction
    predicted_class_name = CLASSES[predicted_class]
    prob_values = [f"{CLASSES[i]}: {probabilities[i]:.4f}" for i in range(len(CLASSES))]
    print(f"Predicted class: {predicted_class_name}")
    print(f"Class probabilities: {', '.join(prob_values)}")
    
    # Generate Grad-CAM for all classes
    results = []
    
    for class_idx, class_name in enumerate(CLASSES):
        # Zero gradients
        model.zero_grad()
        
        # Target for backprop
        one_hot_output = torch.zeros_like(output)
        one_hot_output[0, class_idx] = 1
        
        # Backward pass
        output.backward(gradient=one_hot_output)
        
        # Get gradients and activations
        gradients = model.get_activations_gradient()
        activations = model.get_activations()
        
        # Global average pooling
        weights = torch.mean(gradients, dim=[2, 3])[0, :]
        
        # Create class activation map
        cam = torch.zeros(activations.shape[2:], dtype=torch.float32).to(device)
        
        # Weight the channels by corresponding gradients
        for i, w in enumerate(weights):
            cam += w * activations[0, i, :, :]
        
        # Apply ReLU
        cam = F.relu(cam)
        
        # Normalize
        cam = cam - torch.min(cam)
        if torch.max(cam) > 0:
            cam = cam / torch.max(cam)
        
        # Convert to numpy and resize to match original image
        cam_np = cam.cpu().detach().numpy()
        cam_np = cv2.resize(cam_np, (original_img.shape[1], original_img.shape[0]))
        
        # Convert CAM to heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_np), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        
        # Combine original image with heatmap
        img_float = original_img.astype(float) / 255
        heatmap_float = heatmap.astype(float) / 255
        overlay = 0.7 * img_float + 0.3 * heatmap_float
        overlay = np.clip(overlay, 0, 1)
        
        # Store the results
        results.append((heatmap, overlay, class_name))
        
        # Create a figure to display the results
        if save_results:
            plt.figure(figsize=(12, 5))
            
            plt.subplot(1, 2, 1)
            plt.imshow(original_img)
            plt.title(f"Original Image\nPredicted: {predicted_class_name}")
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(overlay)
            plt.title(f"Grad-CAM: {class_name}\nProbability: {probabilities[class_idx]:.4f}")
            plt.axis('off')
            
            plt.tight_layout()
            
            # Save figure
            output_path = os.path.join(output_dir, f"gradcam_{os.path.basename(image_path)}_{class_name}.png")
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.close()
    
    # Create a combined figure showing all classes
    if save_results:
        plt.figure(figsize=(15, 10))
        
        # Original image
        plt.subplot(2, 3, 1)
        plt.imshow(original_img)
        plt.title(f"Original Image\nPredicted: {predicted_class_name}")
        plt.axis('off')
        
        # Grad-CAM for each class
        for i, (_, overlay, class_name) in enumerate(results):
            plt.subplot(2, 3, i+2)
            plt.imshow(overlay)
            plt.title(f"Grad-CAM: {class_name}\nProbability: {probabilities[i]:.4f}")
            plt.axis('off')
        
        plt.tight_layout()
        
        # Save combined figure
        output_path = os.path.join(output_dir, f"gradcam_{os.path.basename(image_path)}_all_classes.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    return results, original_img

# Example usage
if __name__ == "__main__":
    # Replace with your image path
    image_path = "dataset/Testing/meningioma/Te-me_0017.jpg"
    model_path = "10_Grad_CAM_Results/brain_tumor_model.pth"
    
    # Generate Grad-CAM
    results, original_img = generate_gradcam_for_image(
        image_path=image_path,
        model_path=model_path,
        output_dir="gradcam_test_results"
    )
    
    print("Grad-CAM visualization completed!")

Using device: cpu


/var/folders/cf/vf9kcw4j6zsgy741kz9d9nkm0000gn/T/ipykernel_11241/3118903276.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path,

Predicted class: meningioma
Class probabilities: glioma: 0.0000, meningioma: 0.9980, notumor: 0.0020, pituitary: 0.0000


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [6]:
def generate_gradcam_for_image(image_path, model_path=None, output_dir="gradcam_results", save_results=True):
    """
    Generate Grad-CAM visualizations for a single image across all classes.
    
    Parameters:
    -----------
    image_path : str
        Path to the input image
    model_path : str, optional
        Path to the saved model. If None, uses the default path
    output_dir : str, optional
        Directory to save the results
    save_results : bool, optional
        Whether to save the results to disk
        
    Returns:
    --------
    results : list
        List of tuples (heatmap, overlay, class_name) for each class
    """
    import os
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    import torch
    import torch.nn.functional as F
    import torchvision.transforms as transforms
    from torchvision.models import resnet50
    
    # Create output directory if it doesn't exist and save_results is True
    if save_results:
        os.makedirs(output_dir, exist_ok=True)
    
    # Define classes
    CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
    
    # Set device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Define transformation for the input image
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Create a copy of the original image for display
    original_img = img.copy()
    
    # Define the model with Grad-CAM hooks
    class BrainTumorClassifier(torch.nn.Module):
        def __init__(self, num_classes=4):
            super(BrainTumorClassifier, self).__init__()
            # Load pre-trained ResNet50
            self.model = resnet50(weights=None)  # No pre-trained weights here
            
            # Modify the final fully connected layer
            num_features = self.model.fc.in_features
            self.model.fc = torch.nn.Linear(num_features, num_classes)
            
            # Save the layer names for Grad-CAM
            self.gradients = None
            self.activations = None
            
            # Register hook for the last convolutional layer
            self.model.layer4[-1].conv3.register_forward_hook(self.save_activation)
            self.model.layer4[-1].conv3.register_full_backward_hook(self.save_gradient)
        
        def save_activation(self, module, input, output):
            self.activations = output
        
        def save_gradient(self, module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        def forward(self, x):
            return self.model(x)
        
        def get_activations_gradient(self):
            return self.gradients
        
        def get_activations(self):
            return self.activations
    
    # Load the model
    model = BrainTumorClassifier(num_classes=len(CLASSES)).to(device)
    
    if model_path is None:
        model_path = os.path.join("10_Grad_CAM_Results", "brain_tumor_model.pth")
    
    # Load model with weights_only=True to avoid the FutureWarning
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    
    # Get model prediction - first pass without gradients to get the prediction
    with torch.no_grad():
        input_tensor = transform(img).unsqueeze(0).to(device)
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)[0]
        predicted_class = output.argmax(dim=1).item()
    
    # Print prediction
    predicted_class_name = CLASSES[predicted_class]
    prob_values = [f"{CLASSES[i]}: {probabilities[i]:.4f}" for i in range(len(CLASSES))]
    print(f"Predicted class: {predicted_class_name}")
    print(f"Class probabilities: {', '.join(prob_values)}")
    
    # Generate Grad-CAM for all classes
    results = []
    
    # Need a new forward pass with gradients enabled for Grad-CAM
    input_tensor = transform(img).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    
    for class_idx, class_name in enumerate(CLASSES):
        # Forward pass with gradients
        model.zero_grad()
        outputs = model(input_tensor)
        
        # Target for backprop - one-hot encoding for the current class
        target = torch.zeros(outputs.size()).to(device)
        target[0, class_idx] = 1
        
        # Backward pass
        outputs.backward(gradient=target, retain_graph=True)
        
        # Get gradients and activations
        gradients = model.get_activations_gradient()
        activations = model.get_activations()
        
        if gradients is None or activations is None:
            print(f"Warning: Gradients or activations are None for class {class_name}")
            continue
        
        # Global average pooling
        weights = torch.mean(gradients, dim=[2, 3])[0, :]
        
        # Create class activation map
        cam = torch.zeros(activations.shape[2:], dtype=torch.float32).to(device)
        
        # Weight the channels by corresponding gradients
        for i, w in enumerate(weights):
            cam += w * activations[0, i, :, :]
        
        # Apply ReLU
        cam = F.relu(cam)
        
        # Normalize
        cam = cam - torch.min(cam)
        if torch.max(cam) > 0:
            cam = cam / torch.max(cam)
        
        # Convert to numpy and resize to match original image
        cam_np = cam.cpu().detach().numpy()
        cam_np = cv2.resize(cam_np, (original_img.shape[1], original_img.shape[0]))
        
        # Convert CAM to heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_np), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        
        # Combine original image with heatmap
        img_float = original_img.astype(float) / 255
        heatmap_float = heatmap.astype(float) / 255
        overlay = 0.7 * img_float + 0.3 * heatmap_float
        overlay = np.clip(overlay, 0, 1)
        
        # Store the results
        results.append((heatmap, overlay, class_name))
        
        # Create a figure to display the results
        if save_results:
            plt.figure(figsize=(12, 5))
            
            plt.subplot(1, 2, 1)
            plt.imshow(original_img)
            plt.title(f"Original Image\nPredicted: {predicted_class_name}")
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(overlay)
            plt.title(f"Grad-CAM: {class_name}\nProbability: {probabilities[class_idx]:.4f}")
            plt.axis('off')
            
            plt.tight_layout()
            
            # Save figure
            img_basename = os.path.basename(image_path)
            output_path = os.path.join(output_dir, f"gradcam_{img_basename}_{class_name}.png")
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.close()
    
    # Create a combined figure showing all classes
    if save_results and results:
        plt.figure(figsize=(15, 10))
        
        # Original image
        plt.subplot(2, 3, 1)
        plt.imshow(original_img)
        plt.title(f"Original Image\nPredicted: {predicted_class_name}")
        plt.axis('off')
        
        # Grad-CAM for each class
        for i, (_, overlay, class_name) in enumerate(results):
            plt.subplot(2, 3, i+2)
            plt.imshow(overlay)
            plt.title(f"Grad-CAM: {class_name}\nProbability: {probabilities[i]:.4f}")
            plt.axis('off')
        
        plt.tight_layout()
        
        # Save combined figure
        img_basename = os.path.basename(image_path)
        output_path = os.path.join(output_dir, f"gradcam_{img_basename}_all_classes.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    
    return results, original_img

# Example usage
if __name__ == "__main__":
    # Replace with your image path
    image_path = "dataset/Testing/meningioma/Te-me_0017.jpg"
    model_path = "10_Grad_CAM_Results/brain_tumor_model.pth"
    
    # Generate Grad-CAM
    results, original_img = generate_gradcam_for_image(
        image_path=image_path,
        model_path=model_path,
        output_dir="10_Grad_CAM_Results/gradcam_test_results"
    )
    
    print("Grad-CAM visualization completed!")

Using device: cpu
Predicted class: meningioma
Class probabilities: glioma: 0.0000, meningioma: 0.9980, notumor: 0.0020, pituitary: 0.0000
Grad-CAM visualization completed!
